In [ ]:
# KOREAN-WEBTEXT 데이터셋을 활용한 초급 파이썬 NLP 탐색 실습
# =========================================================================================
# 📚 데이터셋 이름: HAERAE-HUB/KOREAN-WEBTEXT
# ✍️ 데이터 의미: 한국 웹에서 수집된 방대한 양의 고품질 한국어 텍스트 말뭉치입니다.
# ✨ 실습 목표: 이 데이터셋의 구조를 이해하고, 텍스트 길이와 출처 정보를 분석하여
#    AI 모델을 위한 기초적인 데이터 준비 과정을 경험해 봅니다. (난이도: ★☆☆☆☆)

# 🚀 필수 라이브러리 로드
from datasets import load_dataset
import random
import statistics

# 설정 상수
DATASET_NAME = "HAERAE-HUB/KOREAN-WEBTEXT"
SAMPLE_COUNT = 100 # 전체 데이터셋이 너무 크므로, 상위 100개만 샘플링합니다!

print("=======================================================================================")
print(f"🇰🇷 환영합니다, AI 탐험가님! '{DATASET_NAME}' 데이터셋 분석을 시작합니다!")
print("======================================================================================")

# -----------------------------------------------------------------------------
# ⚙️ STEP 1: 데이터셋 로드 및 스트리밍 처리 (가장 까다로운 부분!)
# -----------------------------------------------------------------------------

dataset = None
print("\n[Step 1/3] ☁️ 데이터셋 로드 전략 실행: 스트리밍(Streaming) 모드를 먼저 시도합니다.")

# 1. 스트리밍 모드 시도
try:
    # 스트리밍=True를 사용하여 메모리를 절약하고 빠르게 데이터셋에 접근합니다.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍 모드로 데이터셋을 로드했습니다. 메모리 효율적이에요!")
except Exception as e:
    # 스트리밍 방식이 현재 환경에서 문제가 발생할 경우의 대체 로직
    print(f"⚠️ 스트리밍 로드 실패 감지 ({e.__class__.__name__} 발생). 일반 모드로 fallback 합니다.")
    try:
        # 스트리밍이 불가능할 경우, 작은 수의 샘플만 직접 다운로드하여 로드합니다.
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✅ 성공! 일반 데이터셋 모드로 로드했습니다. (메모리 주의!)")
    except Exception as e_fallback:
        print(f"❌ 치명적 오류: 모든 데이터 로드에 실패했습니다. 오류: {e_fallback}")
        exit()

# -----------------------------------------------------------------------------
# 📝 STEP 2: 샘플 데이터 추출 및 준비 (데이터를 리스트로 변환)
# -----------------------------------------------------------------------------

print("\n[Step 2/3] ✨ 상위 샘플 데이터 준비: AI 실습을 위해 100개의 데이터만 가져옵니다.")

# 💡 주의: 스트리밍 데이터셋은 len() 사용이 불가하므로, take() 함수와 반복문을 사용합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    # take(N)으로 상위 N개의 항목만 가져와서 메모리에 list로 변환합니다.
    sample_iterator = dataset.take(SAMPLE_COUNT)
    sample_data_list = list(sample_iterator)
else:
    # 일반 데이터셋 (Dataset)인 경우
    sample_data_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))

print(f"✨ 준비 완료! 총 {len(sample_data_list)}개의 샘플 데이터를 메모리에 준비했습니다.")


# -----------------------------------------------------------------------------
# 🧪 STEP 3: 데이터 분석 및 AI 실습 (가장 재미있는 부분!)
# -----------------------------------------------------------------------------

# -----------------------------------------
# 🧠 실습 3-1: 정량적 분석 (Token Count 분포)
# -----------------------------------------
print("\n=============================================================")
print("🔬 실습 3-1: [정량 분석] 데이터의 '크기' 분석 (Token Count)")
print("=============================================================")

# 토큰 개수 데이터를 리스트로 추출합니다.
token_counts = [sample['token_count'] for sample in sample_data_list]

if token_counts:
    # 기본적인 통계량 계산
    min_tokens = min(token_counts)
    max_tokens = max(token_counts)
    avg_tokens = statistics.mean(token_counts)
    
    print(f"🔍 샘플 {len(token_counts)}개의 데이터에서 토큰 개수 분포를 확인했습니다.")
    print(f"   - 최소 토큰 수 (Min): {min_tokens} tokens (꽤 짧은 문장!)")
    print(f"   - 최대 토큰 수 (Max): {max_tokens} tokens (길고 풍부한 내용!)")
    print(f"   - 평균 토큰 수 (Mean): {avg_tokens:.2f} tokens (대략 이 정도의 길이입니다.)")
    print("\n💡 [튜터 코멘트] 웹 텍스트 데이터는 길이가 매우 다양합니다. 이 분포를 파악하는 것이 모델 튜닝의 첫걸음이에요!")
else:
    print("😥 분석할 데이터가 부족하여 통계 분석을 건너뜁니다.")


# -----------------------------------------
# 🛠️ 실습 3-2: 텍스트 전처리 및 패턴 인식 (가장 AI스러운 과정!)
# -----------------------------------------
print("\n\n=============================================================")
print("🤖 실습 3-2: [NLP 실습] 텍스트 내용 분석 및 LLM 프롬프트 준비")
print("=============================================================")

# 텍스트 데이터의 특징을 분석하고, 모델 학습에 필요한 정보를 추출해 봅니다.
sample_text_list = [sample['text'] for sample in sample_data_list]

# 첫 3개 샘플을 예시로 분석해 봅시다.
for i in range(min(3, len(sample_text_list))):
    text = sample_text_list[i]
    token_count = sample_data_list[i]['token_count']
    
    # 한국어 텍스트의 복잡도를 간단히 측정해보는 과정 (글자 수 vs 토큰 수)
    char_count = len(text) 
    
    print(f"\n--- [Sample {i+1} 분석] ---")
    print(f"   📝 원문(text) 일부: {text[:50]}...")
    print(f"   📈 토큰 개수: {token_count} tokens")
    print(f"   ✍️ 글자 개수: {char_count} characters")
    
    # LLM 프롬프트 형태로 재구성하는 시뮬레이션
    print("\n    ✨ [🔥 AI Prompt Generation 시뮬레이션]")
    print("    > 당신은 전문 챗봇입니다. 아래 텍스트를 읽고 핵심 내용을 세 가지 요약해주세요.")
    print(f"    > [CONTEXT]: \"{text[:100]}...\"\n")
    
    # (실제 모델 호출 대신, 데이터 구조 파악에 초점을 맞춥니다.)


# -----------------------------------------
# 🌐 실습 3-3: 메타데이터 분류 분석 (Source 분석)
# -----------------------------------------
print("\n\n=============================================================")
print("♻️ 실습 3-3: [분류 분석] 데이터 출처(Source) 분포 확인")
print("=============================================================")

# 출처(Source) 별로 데이터가 얼마나 분포되어 있는지 카운트합니다.
source_counts = {}
for sample in sample_data_list:
    source = sample['source']
    source_counts[source] = source_counts.get(source, 0) + 1

# 가장 많이 사용된 출처 top 3을 출력합니다.
sorted_sources = sorted(source_counts.items(), key=lambda item: item[1], reverse=True)

print("📊 데이터 출처별 등장 빈도 Top 3:")
for i, (source, count) in enumerate(sorted_sources[:3]):
    print(f"   🥇 {i+1}위: '{source}' 출처 - {count} 건 (가장 많은 정보가 담긴 출처!)")

print("\n🎉 축하합니다! 데이터를 성공적으로 탐색했습니다!")
print("이처럼 데이터의 구조, 길이, 출처를 파악하는 과정이 AI 모델 학습의 90%를 차지합니다.")
print("자바스크립트 문법보다 데이터 구조를 이해하는 것이 더 중요할 때가 있답니다! 😉")